In [2]:
mkf_root = '/home/bernardo/Onedrive/Empresa 2'

In [4]:
pwd

'C:\\Users\\leand\\OneDrive\\Empresa 2\\Lab\\medidas de vazamento'

In [6]:
pwd

'C:\\Users\\leand\\OneDrive\\Empresa 2\\Lab\\medidas de vazamento'

In [8]:
import os
#mkf_root = os.getenv("MKF_ROOT")
report_dir = "../medidas de vazamento/"
raw_data_dir = "../Raw Data/Medidas de vazamento/"

In [5]:
import os
#mkf_root = os.getenv("MKF_ROOT")
report_dir = "../Medidas de Ruído de Fase - Laser/"
raw_data_dir = "../Raw Data/Medidas de Ruído de Fase - Laser/"

In [37]:
import time
import numpy as np
import mkf
import importlib
import pickle
from scipy.signal import get_window, convolve, butter, sosfilt_zi, sosfilt
importlib.reload(mkf)
import matplotlib.pyplot as plt
import os
from scipy.signal import find_peaks
%matplotlib qt




def get_calibration_params(wv, figure_n=100, cla=True):
    if len(wv.shape) == 3:
        wv = wv[:,:,0].T
    plt.figure(figure_n)
    if cla:
        plt.cla()
    ellipse_param = mkf.fit_ellipse(*wv)
    t = np.linspace(0,2*np.pi, 200)
    fitted_ellipse = mkf.rescale(np.sin(t), np.cos(t), ellipse_param, invert=True)
    x,y = mkf.rescale(*wv, ellipse_param)
    plt.scatter(*wv)
    plt.plot(*fitted_ellipse, color='red')
    y_inc = 1000/2**13
    print(f"CH1 DC = {ellipse_param[0]*y_inc:.0f} mV - min={min(wv[0])} max={max(wv[0])} DRU={100*max(wv[0])/2**13:.0f}%")
    print(f"CH2 DC = {ellipse_param[1]*y_inc:.0f} mV - min={min(wv[1])} max={max(wv[1])} DRU={100*max(wv[1])/2**13:.0f}%")
    print(f"Radius = {ellipse_param[3]*y_inc:.0f} mV")
    print(f"Eccentricity = {ellipse_param[2]:.2f}")
    print(f"Angle = {ellipse_param[4]*360/(2*np.pi):.1f}")
    return ellipse_param

def demodulate(data):
    N=100
    waveform_len = data.waveforms.shape[-1]
    step_size = waveform_len//N
    calibration_set = data.waveforms[:,::step_size]
    data.ellipse_params = get_calibration_params(calibration_set)
    data.demodulated = mkf.demodulate(data.waveforms, data.ellipse_params)
    data.t = np.arange(waveform_len)/data.sample_frequency_effective

def filter_signal(data, band=[1000, 500_000], order=4):
    data.filter_band = band
    data.filter_order = order
    sos = butter(order, band, 'bandpass', analog=False, fs=data.sample_frequency_effective,  output='sos')
    zi = sosfilt_zi(sos)
    middle = np.average(data.demodulated[:100])
    data.filtered, z2 = sosfilt(sos, data.demodulated, zi=middle*zi)

def rms(data, target="filtered",window_len = 0.1):
    data.rms_window_len = window_len # in seconds
    data.n_rms = int(window_len*data.sample_frequency_effective)
    data.rms = np.sqrt(convolve(data[target]**2, np.ones(data.n_rms)/data.n_rms, mode='valid')[::data.n_rms])
    data.t_rms = np.arange(data.rms.shape[0])*data.n_rms/data.sample_frequency_effective

def avg_slices(function, data, n_slices):
    data_len = data.shape[-1]//n_slices
    res_len = len(function(data[:data_len]))
    matrix = np.empty((n_slices, res_len))
    print(matrix.shape)
    for i in range(n_slices):
        matrix[i] = function(data[i*data_len:(i+1)*data_len])
    return np.sum(matrix, axis=0)/n_slices, np.std(matrix, axis=0)

def decimated(data, factor, avg=False):
    function =  lambda x: fft(x, dB=False)
    n_slices = 100
    if avg:
        decimated = np.sum(data.demodulated.reshape((data.demodulated.shape[-1]//factor,factor)), axis=1)/factor
    else:
        decimated = data.demodulated[::factor]
    wvlen = decimated.shape[0]
    N = wvlen//n_slices
    spectrum, spectrum_std = avg_slices(function, decimated, n_slices)
    f = np.arange(0,N//2)*data.sample_frequency/(N*factor)
    return f, mkf.dB(spectrum)


def reduce_points(vector, factor):
    """
    Reduz o número de pontos em um vetor.
    
    Parâmetros:
        vector (numpy array): O vetor original.
        factor (int): O fator de redução, que determina quantos pontos serão mantidos.
        
    Retorna:
        numpy array: O vetor reduzido.
    """
    if factor < 1:
        raise ValueError("O fator deve ser um inteiro maior ou igual a 1.")
    
    # Seleciona os índices dos pontos a serem mantidos
    indices = np.arange(0, len(vector), factor)
    
    # Retorna o vetor reduzido
    return vector[indices]

def fft(signal, fs=None, t=None, dB=True, window=True, window_type='hamming'):
    N = signal.shape[0]
    if window:
        from scipy.signal import get_window
        signal= signal*get_window(window_type, N)
    ft = np.fft.fft(signal)*(2/N)
    ft = ft[0:N//2]
    ft = np.abs(ft)
    ft = 10*np.log10(ft) if dB else ft
    if t is not None:
        fs = 1/(t[1]-t[0])
    if fs is not None:
        f = np.arange(0,N//2)*fs/N
        return f, ft
    return ft

In [81]:
filenames = list(filter(lambda x: x.endswith('pickle'), os.listdir(raw_data_dir)))
_ = [print(i, name) for i, name in enumerate(filenames)]

0 Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-0.0mm_Vibração-Sim.pickle
1 Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-0.0mm_Vibração-Sim_2.pickle
2 Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-0.0mm_Vibração-Sim_3.pickle
3 Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-0.0mm_Vibração-Sim_4.pickle
4 Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-120.0mm_Vibração-Sim.pickle
5 Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-160.0mm_Vibração-Sim.pickle
6 Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-200.0mm_Vibração-Sim.pickle
7 Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-240.0mm_Vibração-Sim.pickle
8 Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-280.0mm_Vibração-Sim.pickle
9 Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-40.0mm_Vibração-Sim.pickle
10 Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-80.0mm_Vibração-Sim.pickle


In [83]:
data = mkf.dotdict(pickle.load(open(raw_data_dir+filenames[0], 'rb')))

In [85]:
#filenames = [
#"Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-0.0mm.pickle"]
n_slices = 100
data = []
for i, filename in enumerate(filenames):
    data += [mkf.dotdict(pickle.load(open(raw_data_dir+filename, 'rb')))]
    demodulate(data[i])
   # print(data[i].ellipse_params)
    function =  lambda x: fft(x, dB=False, window=True, window_type='blackman')
    data[i].spectrum, data[i].spectrum_std = avg_slices(function, data[i].demodulated, n_slices)
wvlen = data[0].waveforms.shape[-1]
N = wvlen//n_slices
f = np.arange(0,N//2)*data[0].sample_frequency_effective/(N)
    

CH1 DC = 485 mV - min=592 max=7361 DRU=90%
CH2 DC = 410 mV - min=529 max=6208 DRU=76%
Radius = 413 mV
Eccentricity = 1.19
Angle = 28.6
(100, 167772)
CH1 DC = 484 mV - min=590 max=7369 DRU=90%
CH2 DC = 409 mV - min=528 max=6187 DRU=76%
Radius = 413 mV
Eccentricity = 1.20
Angle = 28.4
(100, 167772)
CH1 DC = 485 mV - min=590 max=7389 DRU=90%
CH2 DC = 410 mV - min=522 max=6214 DRU=76%
Radius = 414 mV
Eccentricity = 1.20
Angle = 28.6
(100, 167772)
CH1 DC = 486 mV - min=549 max=7413 DRU=90%
CH2 DC = 409 mV - min=510 max=6178 DRU=75%
Radius = 416 mV
Eccentricity = 1.21
Angle = 28.3
(100, 167772)
CH1 DC = 459 mV - min=410 max=7189 DRU=88%
CH2 DC = 392 mV - min=380 max=6124 DRU=75%
Radius = 414 mV
Eccentricity = 1.20
Angle = 29.1
(100, 167772)
CH1 DC = 419 mV - min=398 max=6758 DRU=82%
CH2 DC = 358 mV - min=371 max=5581 DRU=68%
Radius = 373 mV
Eccentricity = 1.20
Angle = 34.2
(100, 167772)
CH1 DC = 401 mV - min=433 max=6139 DRU=75%
CH2 DC = 359 mV - min=403 max=5465 DRU=67%
Radius = 346 mV
Ecce

In [19]:
plt.figure(4)
plt.cla()

plt.scatter(data[7].waveforms[0,::100],data[7].waveforms[1,::100])
plt.scatter(data[2].waveforms[0,::100],data[2].waveforms[1,::100])

In [261]:
def plot_ellipse(data, figure=100):
    N=100
    waveform_len = data.waveforms.shape[-1]
    step_size = waveform_len//N
    calibration_set = data.waveforms[:,::step_size]
    ellipse_params = get_calibration_params(calibration_set, figure_n=100, cla=False)
plot_ellipse(data[2], figure=100)
plot_ellipse(data[7], figure=100)

CH1 DC = 487 mV - min=654 max=7382 DRU=90%
CH2 DC = 410 mV - min=527 max=5928 DRU=72%
Radius = 413 mV
Eccentricity = 1.20
Angle = 28.5
CH1 DC = 486 mV - min=549 max=7413 DRU=90%
CH2 DC = 409 mV - min=510 max=6178 DRU=75%
Radius = 416 mV
Eccentricity = 1.21
Angle = 28.3


In [241]:
plt.figure(3)
plt.plot(data[7].demodulated)

In [89]:
plt.figure(102)
#plt.cla()
for datum in data[0:]:
    plt.semilogx(f, mkf.dB(datum.spectrum), label=datum.descricao+" - OPD= "+ str(datum.OPD) + ' mm')
    #plt.semilogx(f, mkf.dB(data[i].spectrum_std), label=data[i].descricao)

plt.legend()
plt.title('Espectro do Ruído de Fase')
plt.xlabel('Frequência [Hz]')
plt.ylabel('Amplitude [dB]')
#plt.xlim([100,1E6])
#plt.ylim([-50, -10])
plt.savefig("Espectro do Ruído de Fase", dpi='figure')

In [22]:
plt.figure(103)
plt.cla()
for i in [2, 1]:
    plt.semilogx(f, mkf.dB(data[i].spectrum)-mkf.dB(data[0].spectrum), label=data[i].descricao)
    data[i].spectrum_std
plt.title('Espectro do sinal acima do ruído de fundo')
plt.xlabel('Frequência [Hz]')
plt.ylabel('Amplitude [dB]')
plt.xlim([100,1E6])
plt.ylim([-1, 20])
plt.legend()
plt.savefig("Espectro do sinal acima do ruído de fundo", dpi='figure')

In [80]:
# Plot rms data in the time domain

plt.figure(105)
plt.cla()
processed_data = []
for datum in data:
    filter_signal(datum)
    plt.figure(105)
    plt.plot(datum.t_rms, mkf.dB(datum.rms), label=datum.descricao)
plt.legend()
plt.xlabel("Tempo [s]")
plt.ylabel("Sinal RMS [dB]")
plt.title("Media RMS do sinal acústico. Janela de 100 ms, filtro ordem 4 (1-500 kHz)")
plt.savefig("RMS do sinal", dpi='figure')

AttributeError: rms

In [74]:
bg_raw = mkf.dotdict(pickle.load(open(raw_data_dir+"background_125MSPS.pickle", 'rb')))
demodulate(bg_raw)

CH1 DC = 438 mV - min=549 max=6144 DRU=75%
CH2 DC = 442 mV - min=389 max=4164 DRU=51%
Radius = 382 mV
Eccentricity = 0.97
Angle = 23.9


In [28]:
bg_raw_silencio = mkf.dotdict(pickle.load(open(raw_data_dir+"background_125MSPS_silencio.pickle", 'rb')))
demodulate(bg_raw_silencio)

CH1 DC = 403 mV - min=446 max=6165 DRU=75%
CH2 DC = 406 mV - min=383 max=6286 DRU=77%
Radius = 348 mV
Eccentricity = 0.97
Angle = 24.3


In [29]:
function =  lambda x: mkf.fft(x, dB=False)
n_slices = 100
wvlen = bg_raw.demodulated.shape[0]
N = wvlen//n_slices
bg_raw.spectrum, bg_raw.spectrum_std = avg_slices(function, bg_raw.demodulated, n_slices)
bg_raw.f =  np.arange(0,N//2)*bg_raw.sample_frequency_effective/N

NameError: name 'bg_raw' is not defined

In [ ]:
plt.figure(101)
plt.cla()
plt.semilogx(bg_raw.f , mkf.dB(bg_raw.spectrum), label=bg_raw.descricao)
plt.xlabel("Frequência [Hz]")
plt.ylabel("Densidade espectral de amplitude [dB/Hz]")
plt.xlim([1E3,62.5E6])
plt.ylim([-55, -20])
plt.grid(visible = True, which='both', axis='both')
plt.title("Espectro do sinal acústico de fundo")
plt.savefig("Espectro do sinal acústico de fundo 125 MSPS", dpi='figure')

In [ ]:
plt.xlim([1E6,62.5E6])
plt.ylim([-60, -30])
plt.grid(visible = True, which='both', axis='both')
plt.title("Espectro do sinal acústico de fundo, zoom alta frequencia")
plt.savefig("Espectro do sinal acústico de fundo 125 MSPS, zoom", dpi='figure')

In [130]:
from itertools import product
plt.figure(101)
plt.cla()

for  avg, factor in product([False, True], [1,5,25,125]):
    if avg and factor == 1:
        continue
    print(avg, factor)
    f, spectrum = decimated(data, factor, avg)
    averaged = ', averaged' if avg else ''
    plt.semilogx(f, spectrum, label = f"{125//factor} MSPS{averaged}")
plt.legend()

False 1
False 5
False 25
False 125
True 5
True 25
True 125


In [146]:
plt.figure(111)
plt.cla()

for data in [bg_raw_silencio, bg_raw]:
    avg = True
    factor = 125
    if avg and factor == 1:
        continue
    print(avg, factor)
    f, spectrum = decimated(data, factor, avg)
    averaged = ', averaged' if avg else ''
    plt.semilogx(f, spectrum, label = data.descricao)
plt.legend()

True 125
True 125


In [52]:
def fft(signal, fs=None, t=None, dB=True):
    N = signal.shape[0]
    ft = np.fft.fft(signal)*(2/N)
    ft = ft[0:N//2]
    ft = np.abs(ft)
    ft = 10*np.log10(ft) if dB else ft
    if t is not None:
        fs = 1/(t[1]-t[0])
    if fs is not None:
        f = np.arange(0,N//2)*fs/N
        return f, ft
    return ft

from scipy.signal import get_window
plt.figure(110)
plt.cla()
f = 10
for fs in [2500 ,5000, 10000]:
    t = np.linspace(0,1, fs)
    plt.semilogx(*fft(np.sin(2*np.pi*f*t)*get_window('blackman', fs), fs=fs), label=f'{fs}')
plt.legend

<function matplotlib.pyplot.legend(*args, **kwargs) -> 'Legend'>

# Medidas de Vazamento

In [11]:
#Curva de responsividade

#abrindo dados de responsividade dos sensores

# escolha o sensor: 'Epoxy', 'Poliuretano', 'Nylon', 'Steel', 'Silicone'

#a responsividade em responsividades[sensor][1] é a divisão 
#da resposta do sensor dividido pela resposta do piezo no espaço livre

#Na interpolação os dados já estão em dB!!!

from scipy.interpolate import interp1d

sensor='Poliuretano'

file_name = 'responsividades_V1.pkl'

with open(file_name, 'rb') as file:
    responsividades = pickle.load(file)


plt.figure(figsize=(14,6))
plt.semilogx(responsividades[sensor][0],mkf.dB(responsividades[sensor][1]), color='b')
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude [dB]")
plt.title(f"{sensor} Sensor's Responsivity")

inter_responsividade_sensor = interp1d(responsividades[sensor][0], mkf.dB(responsividades[sensor][1]), kind='quadratic')



FileNotFoundError: [Errno 2] No such file or directory: 'responsividades_V1.pkl'

In [30]:
filenames = list(filter(lambda x: x.endswith('pickle'), os.listdir(raw_data_dir)))
_ = [print(i, name) for i, name in enumerate(filenames)]

0 Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-0.0mm.pickle


In [34]:
file_n = 0
print(f"Loading file: {filenames[file_n]}")
data = mkf.dotdict(pickle.load(open(raw_data_dir+filenames[file_n], 'rb')))

Loading file: Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-0.0mm.pickle


UnpicklingError: pickle data was truncated

In [134]:
filenames = [
#'Vazamento_Plastico Fluxo-2.0L-min Pressão0.3Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico Fluxo-3.0L-min Pressão0.5Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico Fluxo-4.0L-min Pressão2.2Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico Fluxo-5.0L-min Pressão3.4Bar Valvula-closed Material-polyurethane.pickle',
#'Vazamento_Plastico Fluxo-5.0L-min Pressão3.4Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico Fluxo-6.0L-min Pressão4.4Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico Fluxo-7.0L-min Pressão5.4Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico Fluxo-8.0L-min Pressão6.6Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico Fluxo-9.0L-min Pressão7.4Bar Valvula-open Material-polyurethane.pickle'
#'Vazamento_Plastico_furo_3 Fluxo-9.0L-min Pressão4.0Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico_furo_2 Fluxo-2.0L-min Pressão10.0Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico_furo_3 Fluxo-3.0L-min Pressão0.1Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico_furo_4 Fluxo-0.2L-min Pressão0.0Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico_furo_4 Fluxo-0.5L-min Pressão0.2Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico_furo_4 Fluxo-0.8L-min Pressão1.0Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico_furo_4 Fluxo-1.0L-min Pressão1.8Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico_furo_4 Fluxo-1.2L-min Pressão2.2Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico_furo_4 Fluxo-1.5L-min Pressão2.4Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico_furo_3 Fluxo-10.0L-min Pressão5.0Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico_furo_3 Fluxo-12.0L-min Pressão6.2Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico_furo_3 Fluxo-13.0L-min Pressão7.2Bar Valvula-open Material-polyurethane.pickle',
#'Vazamento_Plastico_furo_3 Fluxo-14.0L-min Pressão8.2Bar Valvula-open Material-polyurethane.pickle',
 #   'Vazamento_Plastico_furo_4_com dois sensores Fluxo-1.5L-min Pressão3.0Bar Valvula-open Material-polyurethane.pickle',

#'Vazamento_Plastico_furo_4_1-sensor Fluxo-3.5L-min Pressão8.0Bar Valvula-closed Material-polyurethane Posição-0.0cm.pickle' ,  
#    'Vazamento_Plastico_furo_4_1-sensor Fluxo-3.5L-min Pressão8.0Bar Valvula-open Material-polyurethane Posição-0.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-4.0L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-0.0cm.pickle',
#    'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.5L-min Pressão3.0Bar Valvula-open Material-polyurethane Posição-0.0cm.pickle',
#    'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.0L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-0.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.5L-min Pressão9.5Bar Valvula-open Material-polyurethane Posição-0.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.0L-min Pressão9.5Bar Valvula-open Material-polyurethane Posição-0.0cm.pickle',

#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.0L-min Pressão3.0Bar Valvula-open Material-polyurethane Posição-10.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.5L-min Pressão3.0Bar Valvula-open Material-polyurethane Posição-10.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.0L-min Pressão5.0Bar Valvula-open Material-polyurethane Posição-10.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.5L-min Pressão9.5Bar Valvula-open Material-polyurethane Posição-10.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.5L-min Pressão3.0Bar Valvula-closed Material-polyurethane Posição-10.0cm.pickle',

#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.0L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-20.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.5L-min Pressão3.0Bar Valvula-open Material-polyurethane Posição-20.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.0L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-20.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.0L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-20.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.0L-min Pressão9.0Bar Valvula-closed Material-polyurethane Posição-20.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor_teste com vazamento de ar externo Fluxo-0.0L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-20.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor_teste com vazamento de ar externo Fluxo-1.5L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-20.0cm.pickle'


#'Vazamento_Plastico_furo_4_1-sensor_teste com vazamento de ar externo Fluxo-0.0L-min Pressão3.0Bar Valvula-closed Material-polyurethane Posição-30.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.5L-min Pressão3.0Bar Valvula-open Material-polyurethane Posição-30.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.0L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-30.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.5L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-30.0cm.pickle'


#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.0L-min Pressão3.0Bar Valvula-closed Material-polyurethane Posição-40.0cm.pickle',   
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.0L-min Pressão3.0Bar Valvula-open Material-polyurethane Posição-40.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.5L-min Pressão3.0Bar Valvula-open Material-polyurethane Posição-40.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.0L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-40.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.5L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-40.0cm.pickle',


#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.0L-min Pressão3.0Bar Valvula-closed Material-polyurethane Posição-50.0cm.pickle',   
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.0L-min Pressão3.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.5L-min Pressão3.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.0L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.5L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',

#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.5L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-30.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.5L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-20.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.5L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-40.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.5L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.5L-min Pressão9.5Bar Valvula-open Material-polyurethane Posição-0.0cm.pickle',
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.5L-min Pressão9.5Bar Valvula-open Material-polyurethane Posição-10.0cm.pickle'
#'Vazamento_Plastico_furo_4_2-sensores Fluxo-1.5L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_Plastico_furo_5_2-sensores Fluxo-9.0L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',

#'Vazamento_Plastico_furo_5_2-sensores Fluxo-1.5L-min Pressão1.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
# 'Vazamento_Plastico_furo_5_2-sensores Fluxo-10.5L-min Pressão7.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
# 'Vazamento_Plastico_furo_5_2-sensores Fluxo-12.0L-min Pressão8.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
# 'Vazamento_Plastico_furo_5_2-sensores Fluxo-13.5L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_Plastico_furo_5_2-sensores Fluxo-15.0L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
# 'Vazamento_Plastico_furo_5_2-sensores Fluxo-3.0L-min Pressão2.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
# 'Vazamento_Plastico_furo_5_2-sensores Fluxo-4.5L-min Pressão3.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_Plastico_furo_5_2-sensores Fluxo-6.0L-min Pressão4.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_Plastico_furo_5_2-sensores Fluxo-7.5L-min Pressão5.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_Plastico_furo_5_2-sensores Fluxo-9.0L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_Plastico_furo_5_2-sensores Fluxo-15.0L-min Pressão10.0Bar Valvula-closed Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_Plastico_furo_5_2-sensores Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle' ,   
#'Vazamento_Plastico_furo_5_2-sensores Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-nylon Posição-10.0cm.pickle',
#'Vazamento_Plastico_furo_5_2-sensores Fluxo-0.0L-min Pressão0.0Bar Valvula-closed Material-nylon Posição-10.0cm.pickle',
#'Vazamento_Plastico_furo_5_2-sensores_nas_pontas Fluxo-15.0L-min Pressão10.0Bar Valvula-open Material-nylon Posição-10.0cm.pickle',

    
    #'Vazamento_Plastico_furo_5_2-sensores_nas_pontas Fluxo-15.0L-min Pressão10.0Bar Valvula-open Material-steel Posição-10.0cm.pickle',
#'Vazamento_Plastico_furo_5_2-sensores_nas_pontas Fluxo-0.0L-min Pressão0.0Bar Valvula-closed Material-steel Posição-10.0cm.pickle',
#'Vazamento_Plastico_furo_5_2-sensores_nas_pontas Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-steel Posição-10.0cm.pickle',
#'Vazamento_Plastico_furo_5_1-sensor Fluxo-15.0L-min Pressão10.0Bar Valvula-open Material-steel Posição-10.0cm.pickle'

#'Descargas_Elétricas_1-sensor Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-steel Posição-10.0cm.pickle',
#'Descargas_Elétricas_1-sensor Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-nylon Posição-10.0cm.pickle' ,
# 'Teste_ruido_1-sensor Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-polyurethane Posição-10.0cm.pickle',
#    'Teste_ruido_1-sensor Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-polyurethane Posição--50.0cm.pickle',
#   'Teste_ruido_forte_1-sensor Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-polyurethane Posição--50.0cm.pickle', 
#'Teste_ruido_forte_1-sensor Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-polyurethane Posição--50.0cm muito forte.pickle'

# vazamento com água:

#'Vazamento_agua_2-sensores Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-polyurethane Posição-50.0cm muito forte.pickle',
#    'Vazamento_agua_2-sensores Fluxo-0.0L-min Pressão0.0Bar Valvula-closed Material-polyurethane Posição-50.0cm muito forte.pickle',

#'Vazamento_agua_2-sensores Fluxo-0.5L-min Pressão1.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',

  #  'Vazamento_agua_2-sensores Fluxo-1.0L-min Pressão2.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores Fluxo-2.0L-min Pressão4.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores Fluxo-3.0L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores Fluxo-4.0L-min Pressão8.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#    'Vazamento_agua_2-sensores Fluxo-5.0L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#    'Vazamento_agua_2-sensores_furo_2 Fluxo-0.0490L-min Pressão8.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_2 Fluxo-0.0337L-min Pressão5.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_2 Fluxo-0.0450L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle'   , 
#'Vazamento_agua_2-sensores_furo_3 Fluxo-0.0360L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_3_com_rudio_externo Fluxo-0.0360L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',   
#'Vazamento_agua_2-sensores_furo_3_sem_rudio_externo Fluxo-0.0000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_3_com_rudio_externo Fluxo-0.0000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
 #   'Vazamento_agua_2-sensores_furo_3_sem_rudio_externo Fluxo-0.0000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_3_com_rudio_externo_de_chave_batendo Fluxo-0.0000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_4 Fluxo-0.0040L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#    'Vazamento_agua_2-sensores_furo_4 Fluxo-0.0000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#    'Vazamento_agua_2-sensores_furo_4 Fluxo-0.00450L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle'


#Testes de vazamento de Água Furo 5:

#'Vazamento_agua_2-sensores_furo_5_sem_ruido_ar Fluxo-0.02200L-min Pressão1.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_5_sem_ruido Fluxo-0.00000L-min Pressão1.0Bar Valvula-closed Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_5_sem_ruido Fluxo-0.00000L-min Pressão1.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_5_sem_ruido_ar Fluxo-0.04200L-min Pressão4.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_5_sem_ruido_ar Fluxo-0.05200L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_5_sem_ruido_ar Fluxo-0.05800L-min Pressão8.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_5_sem_ruido_ar Fluxo-0.06300L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_5_sem_ruido_ar Fluxo-0.02600L-min Pressão2.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle'


#Testes de vazamento de Água Furo 6:
#'Vazamento_agua_2-sensores_furo_6_sem_ruido_ar Fluxo-0.00000L-min Pressão10.0Bar Valvula-closed Material-polyurethane Posição-50.0cm.pickle',

#    'Vazamento_agua_2-sensores_furo_6_sem_ruido_ar Fluxo-0.00000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_6_sem_ruido_ar Fluxo-0.00260L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle'

#VAZAMENTO FURO 6

#    'Vazamento_agua_2-sensores_furo_7_sem_ruido_ar Fluxo-0.13500L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_7_sem_ruido_ar_som_756Hz Fluxo-0.13500L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Teste de gravacao de voz + Vazamento 2 Fluxo-0.13500L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle'
#
#'Vazamento_agua_2-sensores_furo_7_sem_ruido_ar_som_756Hz_2 Fluxo-0.13500L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
 #   'Vazamento_agua_2-sensores_furo_7_sem_ruido_ar_som_756Hz_3 Fluxo-0.13500L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
      #  'Vazamento_agua_2-sensores_furo_7_sem_ruido_ar_som_756Hz_4 Fluxo-0.13500L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle'


#Teste Tubo de Metal

   # 'Tubo_Metal_Vazamento_Gas_Teste_correlacao Fluxo-1.50000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-400.0cm.pickle',
#'Tubo_Metal_Vazamento_Gas_Teste_correlacao_Apenas_1_sensor_a_4m Fluxo-10.00000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-400.0cm.pickle',   
#'Tubo_Metal_Vazamento_Gas_Teste_correlacao_dois_sensores_a_4m Fluxo-10.00000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-400.0cm.pickle',
#'Tubo_Metal_Vazamento_Gas_Teste_correlacao_apenas_um_sensor_a_2cm Fluxo-10.00000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-2.0cm.pickle',
#'Tubo_Metal_Vazamento_Gas_Teste_correlacao_apenas_um_sensor_a_400cm Fluxo-10.00000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-400.0cm.pickle',
#'Tubo_Metal_Vazamento_Gas_Teste_correlacao_apenas_um_sensor_a_400cm_teste Fluxo-10.00000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-400.0cm.pickle'    

#Teste sensor 1 vs sensor 2 de poliuretano
'Tubo_Metal_Vazamento_Gas_Teste_sensor_1+2_teste-outocorrelação_na_posição_invertida Fluxo-10.00000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-400.0cm.pickle',
'Tubo_Metal_teste_na_mesma_posicao Fluxo-10.00000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-2.0cm.pickle',   
'Tubo_Metal_Vazamento_Gas_Teste_sensor_1 Fluxo-10.00000L-min Pressão10.0Bar Valvula-closed Material-polyurethane Posição-2.0cm.pickle',
'Tubo_Metal_Vazamento_Gas_Teste_sensor_1 Fluxo-10.00000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-2.0cm.pickle',
'Tubo_Metal_Vazamento_Gas_Teste_sensor_2 Fluxo-10.00000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-2.0cm.pickle',
'Tubo_Metal_Vazamento_Gas_Teste_sensor_1+2_teste-outocorrelação Fluxo-10.00000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-400.0cm.pickle'

]

def media_movel(dados, janela):

    return np.convolve(dados, np.ones(janela)/janela, mode='valid')
n_slices = 100



data = []
for i, filename in enumerate(filenames):
    data += [mkf.dotdict(pickle.load(open(raw_data_dir+filename, 'rb')))]
    demodulate(data[i])
    function =  lambda x: mkf.fft(x*get_window('blackman', x.shape[-1]), dB=False)
    data[i].spectrum, data[i].spectrum_std = avg_slices(function, data[i].demodulated, n_slices)
wvlen = data[0].waveforms.shape[-1]
N = wvlen//n_slices
f = np.arange(0,N//2)*data[0].sample_frequency_effective/(N)

FileNotFoundError: [Errno 2] No such file or directory: '../Raw Data/Medidas de vazamento/Tubo_Metal_Vazamento_Gas_Teste_sensor_1+2_teste-outocorrelação_na_posição_invertida Fluxo-10.00000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-400.0cm.pickle'

In [131]:
plt.figure()
for datum in data:
    if datum.valve_state == 'closed':
        ref= datum.spectrum[11:-1]
  
for datum in data:    
    if datum.valve_state != 'closed':
        plt.semilogx(f[11:-1], mkf.dB(datum.spectrum[11:-1])-mkf.dB(ref), label=f"{datum.descricao} {datum.flow:.4f}{datum.flow_unit} {datum.preassure:.1f}{datum.preassure_unit} {datum.valve_state} {datum.sensor} Posição-{datum.position}{datum.position_unit}")
        #plt.semilogx(f[11:-1], datum.spectrum[11:-1]/ref, label=f"{datum.descricao} {datum.flow:.4f}{datum.flow_unit} {datum.preassure:.1f}{datum.preassure_unit} {datum.valve_state} {datum.sensor} Posição-{datum.position}{datum.position_unit}")
        
plt.legend()
plt.title('Espectro do sinal')
plt.xlabel('Frequência [Hz]')
plt.ylabel('Amplitude [dB]')
#plt.ylabel('Sinal Relativo (u.a.)')
#plt.xlim([100,1E6])
#plt.ylim([-2, 13])
plt.savefig("Espectro do sinal de vazamento", dpi='figure')

In [49]:
plt.figure(102)
plt.cla()

for datum in data:
    plt.plot(reduce_points(datum.t,100),reduce_points(datum.demodulated,100), label=f"{datum.descricao} {datum.flow:.1f}{datum.flow_unit} {datum.preassure:.1f}{datum.preassure_unit} {datum.valve_state} {datum.sensor} Posição-{datum.position}{datum.position_unit}")

plt.legend()
plt.title('Amostra do Sinal de Vazamento')
plt.xlabel('Tempo [s]')
plt.ylabel('Amplitude [rad]')
#plt.xlim([100,1E6])
#plt.ylim([-2, 13])
plt.show

<function matplotlib.pyplot.show(*, block=None)>

In [152]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

#arquivo usado:
#'Vazamento_Plastico_furo_5_2-sensores Fluxo-9.0L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle'    


amplitude= mkf.dB(datum.spectrum[11:-1])-mkf.dB(ref)
frequency=f[11:-1]

peaks, _ = find_peaks(amplitude, height=0, distance=200)  # Ajustar 'height' conforme necessário para sensibilidade

#ressonancias no tubo de 56cm:
freqs= np.array([ 44.277e3  ])


# Criar o gráfico
plt.figure(figsize=(10, 6))
plt.plot(frequency, amplitude, label='Espectro', color='blue')
plt.scatter(frequency[peaks], amplitude[peaks], color='red', marker='x', label='Ressonâncias medidas')

for freq in freqs:
        plt.axvline(x=freq, color='red', linestyle='--', linewidth=1)
        plt.text(freq, max(amplitude) * 0.95, f'{freq:.2f} Hz', 
                 fontsize=9, color='red', ha='center', va='bottom', rotation=90)

# Adicionar anotações para cada ressonância encontrada
for peak in peaks:
    plt.text(frequency[peak], amplitude[peak], f'{frequency[peak]:.2f} Hz', 
             fontsize=9, ha='center', va='bottom')

# Ajustar rótulos e título do gráfico
plt.xlabel('Frequência (Hz)')
plt.ylabel('Amplitude')
plt.xlim([0,50e3])
plt.title('Espectro com Ressonâncias Destacadas')
plt.legend()
plt.grid(True)

# Mostrar o gráfico
plt.show()

In [85]:
len(datum.t)

20000000

In [12]:
#estudo das frequencias de ressonancia

In [78]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

#arquivo usado:
#'Vazamento_Plastico_furo_5_2-sensores Fluxo-9.0L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle'    


amplitude= mkf.dB(datum.spectrum[11:-1])-mkf.dB(ref)
frequency=f[11:-1]

peaks, _ = find_peaks(amplitude, height=0, distance=20)  # Ajustar 'height' conforme necessário para sensibilidade

#ressonancias no tubo de 56cm:
freqs= np.array([  306.25,   612.5 ,   918.75,  1225.  ,  1531.25,  1837.5 ,
        2143.75,  2450.  ,  2756.25,  3062.5 ,  3368.75,  3675.  ,
        3981.25,  4287.5 ,  4593.75,  4900.  ,  5206.25,  5512.5 ,
        5818.75,  6125.  ,  6431.25,  6737.5 ,  7043.75,  7350.  ,
        7656.25,  7962.5 ,  8268.75,  8575.  ,  8881.25,  9187.5 ,
        9493.75,  9800.  , 10106.25, 10412.5 , 10718.75, 11025.  ,
       11331.25, 11637.5 , 11943.75, 12250.  ])


# Criar o gráfico
plt.figure(figsize=(10, 6))
plt.plot(frequency, amplitude, label='Espectro', color='blue')
plt.scatter(frequency[peaks], amplitude[peaks], color='red', marker='x', label='Ressonâncias medidas')

for freq in freqs:
        plt.axvline(x=freq, color='red', linestyle='--', linewidth=1)
        plt.text(freq, max(amplitude) * 0.95, f'{freq:.2f} Hz', 
                 fontsize=9, color='red', ha='center', va='bottom', rotation=90)

# Adicionar anotações para cada ressonância encontrada
for peak in peaks:
    plt.text(frequency[peak], amplitude[peak], f'{frequency[peak]:.2f} Hz', 
             fontsize=9, ha='center', va='bottom')

# Ajustar rótulos e título do gráfico
plt.xlabel('Frequência (Hz)')
plt.ylabel('Amplitude')
plt.xlim([0,10e3])
plt.title('Espectro com Ressonâncias Destacadas')
plt.legend()
plt.grid(True)

# Mostrar o gráfico
plt.show()

# Estudo da densidade de energia das assinaturas de ultrassom:


In [37]:
import numpy as np

def integrar_energia(frequencias, densidade_espectral, freq_min, freq_max):
    # Filtra os valores dentro do intervalo de frequências especificado
    mask = (frequency[peak] >= freq_min) & (frequencias <= freq_max)
    
    # Frequências e densidade espectral dentro do intervalo
    freqs_filtradas = frequencias[mask]
    densidade_filtrada = densidade_espectral[mask]
    
    # Integra a densidade espectral no intervalo de frequências usando a regra do trapézio
    energia = np.trapz(densidade_filtrada, freqs_filtradas)
    
    return energia


def media_movel(dados, janela):

    return np.convolve(dados, np.ones(janela)/janela, mode='valid')

In [122]:
plt.figure(106)

freq_min=12_000
freq_max=15_000

#freq_min=22_000
#freq_max=28_000

#freq_min=28_000
#freq_max=42_000

#freq_min=42_000
#freq_max=48_000

#freq_min=48_000
#freq_max=100_000

#freq_min=100
#freq_max=100_000


fluxos=[]
energias=[]



for datum in data:
    if datum.valve_state == 'closed':
        
        
        ref= datum.spectrum[11:-1]
        energias_fundo=np.array([integrar_energia(f[11:-1], datum.spectrum[11:-1]/ref, freq_min, freq_max)])
    


for datum in data:    
    if datum.valve_state != 'closed':       
        
        energias.append(integrar_energia(f[11:-1], datum.spectrum[11:-1]/ref, freq_min, freq_max))
        fluxos.append(datum.flow)


 
plt.plot(fluxos,energias-energias_fundo[0],'o',label=f"Banda de integração: {freq_min:.1f} à {freq_max:.1f} Hz")
plt.legend()
plt.title('Integral da Energia acústica para diferentes Bandas')
plt.xlabel('Vazão [L/min]')
plt.ylabel('Integral da Amplitude [ua]')
#plt.xlim([100,1E6])
#plt.ylim([-2, 13])
plt.show()

In [138]:
plt.plot(datum.spectrum[11:-1])

# Experimento Wavelets


In [ ]:
#'Vazamento_Plastico_furo_4_1-sensor_teste com vazamento de ar externo Fluxo-1.5L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-20.0cm.pickle'
#'Vazamento_Plastico_furo_4_1-sensor_teste com vazamento de ar externo Fluxo-0.0L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-20.0cm.pickle',


In [45]:
filenames = [
#'Vazamento_Plastico_furo_4_1-sensor_teste com vazamento de ar externo Fluxo-1.5L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-20.0cm.pickle'
#'Vazamento_Plastico_furo_4_1-sensor_teste com vazamento de ar externo Fluxo-0.0L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-20.0cm.pickle',
  'Vazamento_agua_2-sensores Fluxo-5.0L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
# 'Vazamento_agua_2-sensores_furo_2 Fluxo-0.0490L-min Pressão8.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_2 Fluxo-0.0337L-min Pressão5.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_2 Fluxo-0.0450L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle'    
   
]

def media_movel(dados, janela):

    return np.convolve(dados, np.ones(janela)/janela, mode='valid')
n_slices = 100



data = []
for i, filename in enumerate(filenames):
    data += [mkf.dotdict(pickle.load(open(raw_data_dir+filename, 'rb')))]
    demodulate(data[i])
    function =  lambda x: mkf.fft(x*get_window('blackman', x.shape[-1]), dB=False)
    data[i].spectrum, data[i].spectrum_std = avg_slices(function, data[i].demodulated, n_slices)
wvlen = data[0].waveforms.shape[-1]
N = wvlen//n_slices
f = np.arange(0,N//2)*data[0].sample_frequency_effective/(N)

CH1 DC = 454 mV - min=446 max=7005 DRU=86%
CH2 DC = 475 mV - min=425 max=7432 DRU=91%
Radius = 399 mV
Eccentricity = 0.94
Angle = 24.7
(100, 100000)


In [46]:
import numpy as np
import matplotlib.pyplot as plt
import pywt
from scipy.io import wavfile  # Exemplo de leitura de arquivo WAV

# Carregar os dados (substitua pelo seu próprio método de carregar o arquivo)
# Exemplo para carregar um arquivo .wav:
sample_rate, data = data[0].sample_frequency_effective, data[0].demodulated[0:10000]


# Normalizar o sinal se necessário
data = data / np.max(np.abs(data))

# Definir a faixa de frequências do ruído de vazamento
vazamento_f_min = 13000  # 13 kHz
vazamento_f_max = 45000  # 45 kHz
f_max_visualizacao = 100000  # Limitar a visualização a 100 kHz

# Definir as wavelets e escalas para a varredura
wavelets = ['cmor', 'morl', 'mexh']  # Exemplos: Complex Morlet, Morlet, Mexican hat
scales_range = [np.arange(1, 64), np.arange(1, 128), np.arange(1, 256)]  # Diferentes faixas de escala

# Criar uma figura 2D
fig, ax = plt.subplots(figsize=(10, 8))

# Loop para realizar varredura sobre diferentes wavelets e escalas
for wavelet, scales in zip(wavelets, scales_range):
    coefficients, frequencies = pywt.cwt(data, scales, wavelet, sampling_period=1/sample_rate)
    
    # Filtrar frequências acima de 100 kHz
    freq_idx = np.where(frequencies <= f_max_visualizacao)[0]
    filtered_coefficients = coefficients[freq_idx, :]
    filtered_frequencies = frequencies[freq_idx]
    
    # Gerar um gráfico 2D para as frequências até 100 kHz
    time = np.linspace(0, len(data) / sample_rate, num=len(data))
    
    # Exibir a magnitude dos coeficientes em um gráfico 2D
    ax.imshow(np.abs(filtered_coefficients), extent=[0, len(data) / sample_rate, filtered_frequencies[-1], filtered_frequencies[0]],
              aspect='auto', cmap='jet', vmax=np.percentile(np.abs(filtered_coefficients), 99))

# Adicionar labels e título
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('Análise Wavelet Contínua - Varredura de Wavelets (Frequências até 100 kHz)')

# Mostrar o gráfico 2D
plt.colorbar(label='Magnitude')
plt.show()


RuntimeError: No mappable was found to use for colorbar creation. First define a mappable such as an image (with imshow) or a contour set (with contourf).

# TEstes de Correlação


In [34]:
#Cálculo da autocorrelação

from scipy.signal import correlate


def autocorrelacao_scipy(x,y):
    autocorr = correlate(x, y, mode='full')  # Calcular a autocorrelação completa
    return autocorr 

In [1]:
filenames = [
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-0.0L-min Pressão3.0Bar Valvula-closed Material-polyurethane Posição-40.0cm.pickle',   
#'Vazamento_Plastico_furo_4_1-sensor Fluxo-1.5L-min Pressão9.5Bar Valvula-open Material-polyurethane Posição-10.0cm.pickle'
#'Vazamento_Plastico_furo_4_2-sensores Fluxo-1.5L-min Pressão9.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle'
#'Vazamento_Plastico_furo_5_2-sensores Fluxo-9.0L-min Pressão6.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle'    

 #   'Vazamento_Plastico_furo_5_2-sensores Fluxo-0.0L-min Pressão0.0Bar Valvula-closed Material-nylon Posição-10.0cm.pickle',
#'Vazamento_Plastico_furo_5_2-sensores_nas_pontas Fluxo-15.0L-min Pressão10.0Bar Valvula-open Material-nylon Posição-10.0cm.pickle'


#'Vazamento_Plastico_furo_5_2-sensores_nas_pontas Fluxo-0.0L-min Pressão0.0Bar Valvula-closed Material-steel Posição-10.0cm.pickle',
#'Vazamento_Plastico_furo_5_2-sensores_nas_pontas Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-steel Posição-10.0cm.pickle',
#'Vazamento_Plastico_furo_5_1-sensor Fluxo-15.0L-min Pressão10.0Bar Valvula-open Material-steel Posição-10.0cm.pickle'

#'Descargas_Elétricas_1-sensor Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-steel Posição-10.0cm.pickle',
   # 'Descargas_Elétricas_1-sensor Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-nylon Posição-10.0cm.pickle'    
 
    # vazamento com água:
    
 #   'Vazamento_agua_2-sensores_furo_2 Fluxo-0.0490L-min Pressão8.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_2 Fluxo-0.0337L-min Pressão5.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',
#'Vazamento_agua_2-sensores_furo_2 Fluxo-0.0450L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-50.0cm.pickle',    


#'Vazamento_agua_2-sensores Fluxo-0.0L-min Pressão0.0Bar Valvula-open Material-polyurethane Posição-50.0cm muito forte.pickle',
   'Vazamento_agua_2-sensores Fluxo-0.0L-min Pressão0.0Bar Valvula-closed Material-polyurethane Posição-50.0cm muito forte.pickle',
      
'Tubo_Metal_Vazamento_Gas_Teste_correlacao_dois_sensores_a_4m Fluxo-10.00000L-min Pressão10.0Bar Valvula-open Material-polyurethane Posição-400.0cm.pickle'
]


n_slices = 100
data = []
for i, filename in enumerate(filenames):
    data += [mkf.dotdict(pickle.load(open(raw_data_dir+filename, 'rb')))]
    demodulate(data[i])
    function =  lambda x: mkf.fft(x*get_window('blackman', x.shape[-1]), dB=False)
    data[i].spectrum, data[i].spectrum_std = avg_slices(function, data[i].demodulated, n_slices)
wvlen = data[0].waveforms.shape[-1]
N = wvlen//n_slices
f = np.arange(0,N//2)*data[0].sample_frequency_effective/(N)


NameError: name 'mkf' is not defined

In [18]:
plt.plot(data[-1].t, data[-1].demodulated, label=f"{data[1].descricao} {data[1].flow:.1f}{data[1].flow_unit} {data[1].preassure:.1f}{data[1].preassure_unit} {data[1].valve_state} {data[1].sensor} Posição-{data[1].position}{data[1].position_unit}")

In [29]:

import scipy.signal as filter

sample_frequency_eff= 1_953_125

band = [100, 250_000]
sos = filter.butter(10, band, 'bandpass', analog=False, fs=sample_frequency_eff,  output='sos')
zi = filter.sosfilt_zi(sos)
middle = np.average(data[-1].demodulated)
filtered, z2 = filter.sosfilt(sos, data[-1].demodulated, zi=middle*zi)
plt.figure(104)
plt.cla()
#plt.plot(t, filtered, label=f"{datum.descricao} {datum.flow:.1f}{datum.flow_unit} {datum.preassure:.1f}{datum.preassure_unit} {datum.valve_state} {datum.sensor} Posição-{datum.position}{datum.position_unit}")
plt.plot(data[1].t, filtered, label=f"{data[1].descricao} {data[1].flow:.1f}{data[1].flow_unit} {data[1].preassure:.1f}{data[1].preassure_unit} {data[1].valve_state} {data[1].sensor} Posição-{data[1].position}{data[1].position_unit}")

In [22]:
function =  lambda x: mkf.fft(x*get_window('blackman', x.shape[-1]), dB=False)
FFT, FFT_std=avg_slices(function, filtered, n_slices)

wvlen = data[0].waveforms.shape[-1]
N = wvlen//n_slices
f = np.arange(0,N//2)*data[0].sample_frequency_effective/(N)

(100, 100000)


In [23]:
plt.semilogx(f, FFT)

In [168]:
plt.plot(f, mkf.dB(FFT))

In [170]:
plt.figure(102)
plt.cla()
for datum in data:
    if datum.valve_state == 'closed':
        ref= datum.spectrum[11:-1]
  
for datum in data:    
    if datum.valve_state != 'closed':
        plt.semilogx(f[11:-1], mkf.dB(datum.spectrum[11:-1])-mkf.dB(ref), label=f"{datum.descricao} {datum.flow:.1f}{datum.flow_unit} {datum.preassure:.1f}{datum.preassure_unit} {datum.valve_state} {datum.sensor} Posição-{datum.position}{datum.position_unit}Hz")
plt.legend()
plt.title('Espectro do sinal')
plt.xlabel('Frequência [Hz]')
plt.ylabel('Amplitude [dB]')
plt.xlim([100,1E6])
plt.ylim([-2, 20])
plt.savefig("Espectro do sinal de vazamento", dpi='figure')


In [52]:
plt.figure(103)
plt.cla()
for datum in data:
    if datum.valve_state == 'closed':
        ref= datum.spectrum[11:-1]
  
for datum in data:    
    if datum.valve_state != 'closed':
        plt.semilogx(f, FFT, label=f"{datum.descricao} {datum.flow:.1f}{datum.flow_unit} {datum.preassure:.1f}{datum.preassure_unit} {datum.valve_state} {datum.sensor} Posição-{datum.position}{datum.position_unit} Filtro: {band}")
plt.legend()
plt.title('Espectro do sinal de vazamento filtrado')
plt.xlabel('Frequência [Hz]')
plt.ylabel('Amplitude [dB]')
#plt.xlim([100,1E6])
#plt.ylim([-2, 20])
plt.savefig("Espectro do sinal de vazamento filtrado", dpi='figure')


In [19]:
plt.plot(data[1].t, filtered, label=f"{data[1].descricao} {data[1].flow:.1f}{data[1].flow_unit} {data[1].preassure:.1f}{data[1].preassure_unit} {data[1].valve_state} {data[1].sensor} Posição-{data[1].position}{data[1].position_unit}")

In [40]:
plt.figure(105)
#plt.cla()
sinal=filtered[1000:-1000]
#sinal=data[-1].demodulated[1000:-1000]


t=data[1].t[1000:-1000]
#Number_points= len(sinal)
#sample_frequency_eff= 1_953_125
#t=np.arange(0, Number_points)/sample_frequency_eff
encurtamento=1
size=(int(len(sinal)/encurtamento))
#sinal=sinal[int(size/2)-int(size/4):int(size/2)+int(size/4)]
#size=10000
corr1=correlate(sinal[0:size],sinal[0:size] , mode='same')
#corr2=correlate(sinal[0:size],-sinal[0:size] , mode='same')
t_c=t[0:size]
#corr=correlate(sinal[0:size],sinal[0:size] , mode='full')

#corr_c=corr[int(size/2)-int(size/4):int(size/2)+int(size/4)]
#t_c=t[int(size/2)-int(size/4):int(size/2)+int(size/4)-1]


#plt.plot(t_c,corr1, label=f"{datum.descricao} {datum.flow:.1f}{datum.flow_unit} {datum.preassure:.1f}{datum.preassure_unit} {datum.valve_state} {datum.sensor}")
plt.plot(corr1, label=f"{datum.descricao} {datum.flow:.1f}{datum.flow_unit} {datum.preassure:.1f}{datum.preassure_unit} {datum.valve_state} {datum.sensor}")
#plt.plot(t_c,corr2, label=f"{datum.descricao} {datum.flow:.1f}{datum.flow_unit} {datum.preassure:.1f}{datum.preassure_unit} {datum.valve_state} {datum.sensor}")

plt.legend()
plt.title('Autocorrelação do Sinal Demodulado')
plt.xlabel('Tempo [s]')
plt.ylabel('Amplitude')
#plt.xlim([0.9,1.1])
#plt.ylim([-2, 13])
#plt.savefig("Autocorrelação do Sinal Demodulado", dpi='figure')

Text(0, 0.5, 'Amplitude')